In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from scipy.signal import butter, filtfilt, savgol_filter, welch
import pywt
from pykalman import KalmanFilter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("  Signal Processing Module Initialized")

## 1. Generate Synthetic Market Data

In [ ]:
# Generate realistic price data
np.random.seed(42)
n_samples = 2000
t = np.linspace(0, 10, n_samples)

# Base trend
trend = 100 + 0.5 * t

# Cyclical components (market cycles)
cycle_slow = 2 * np.sin(2 * np.pi * 0.1 * t)
cycle_fast = 1 * np.sin(2 * np.pi * 0.5 * t)

# Random walk component
random_walk = np.cumsum(np.random.randn(n_samples) * 0.1)

# Market microstructure noise
noise = np.random.randn(n_samples) * 0.5

# Composite price signal
true_price = trend + cycle_slow + cycle_fast + random_walk
observed_price = true_price + noise

# Returns
returns = np.diff(observed_price) / observed_price[:-1]

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

axes[0].plot(t, observed_price, 'b-', alpha=0.5, linewidth=0.8, label='Observed')
axes[0].plot(t, true_price, 'r-', linewidth=1.5, label='True')
axes[0].set_ylabel('Price')
axes[0].set_title('Synthetic Market Price Data')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t[1:], returns, 'g-', alpha=0.7, linewidth=0.5)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Returns')
axes[1].set_title('Price Returns')
axes[1].grid(True, alpha=0.3)

axes[2].hist(returns, bins=50, density=True, alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Returns')
axes[2].set_ylabel('Density')
axes[2].set_title('Returns Distribution')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Signal-to-Noise Ratio: {true_price.std() / noise.std():.2f}")

## 2. Digital Filters

In [ ]:
# Butterworth low-pass filter
def apply_lowpass_filter(data, cutoff_freq, sampling_rate, order=4):
    nyquist = sampling_rate / 2
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    filtered_data = filtfilt(b, a, data)
    return filtered_data

# Apply filters with different cutoffs
sampling_rate = n_samples / 10  # Samples per unit time

price_filtered_low = apply_lowpass_filter(observed_price, cutoff_freq=5, 
                                          sampling_rate=sampling_rate, order=4)
price_filtered_high = apply_lowpass_filter(observed_price, cutoff_freq=20, 
                                           sampling_rate=sampling_rate, order=4)

# Savitzky-Golay filter (preserves peaks)
price_savgol = savgol_filter(observed_price, window_length=51, polyorder=3)

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Full view
axes[0].plot(t, observed_price, 'gray', alpha=0.3, linewidth=0.5, label='Original')
axes[0].plot(t, price_filtered_low, 'b-', linewidth=2, label='Butterworth (cutoff=5Hz)')
axes[0].plot(t, price_savgol, 'r-', linewidth=2, alpha=0.7, label='Savitzky-Golay')
axes[0].set_ylabel('Price')
axes[0].set_title('Filtered Price Signals')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zoomed view
zoom_start, zoom_end = 500, 700
axes[1].plot(t[zoom_start:zoom_end], observed_price[zoom_start:zoom_end], 
             'gray', alpha=0.5, linewidth=1, label='Original')
axes[1].plot(t[zoom_start:zoom_end], price_filtered_low[zoom_start:zoom_end], 
             'b-', linewidth=2, label='Butterworth')
axes[1].plot(t[zoom_start:zoom_end], price_savgol[zoom_start:zoom_end], 
             'r-', linewidth=2, alpha=0.7, label='Savitzky-Golay')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Price')
axes[1].set_title('Filtered Signals (Zoomed)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate filtering errors
error_butterworth = np.mean((price_filtered_low - true_price)**2)
error_savgol = np.mean((price_savgol - true_price)**2)

print(f"MSE Butterworth: {error_butterworth:.4f}")
print(f"MSE Savitzky-Golay: {error_savgol:.4f}")

## 3. Wavelet Analysis

In [ ]:
# Wavelet decomposition
wavelet = 'db4'  # Daubechies wavelet
level = 5

# Discrete wavelet transform
coeffs = pywt.wavedec(observed_price, wavelet, level=level)

# Reconstruct components at each level
reconstructed = []
for i in range(len(coeffs)):
    coeff_list = [np.zeros_like(c) if j != i else c for j, c in enumerate(coeffs)]
    recon = pywt.waverec(coeff_list, wavelet)
    # Trim to original length
    reconstructed.append(recon[:len(observed_price)])

# Visualization
fig, axes = plt.subplots(level + 2, 1, figsize=(14, 16))

axes[0].plot(t, observed_price, 'b-', linewidth=0.5)
axes[0].set_title('Original Signal')
axes[0].set_ylabel('Price')
axes[0].grid(True, alpha=0.3)

for i, recon in enumerate(reconstructed):
    axes[i+1].plot(t, recon, linewidth=1)
    if i == 0:
        axes[i+1].set_title(f'Approximation (Level {level})')
    else:
        axes[i+1].set_title(f'Detail (Level {level - i + 1})')
    axes[i+1].set_ylabel('Amplitude')
    axes[i+1].grid(True, alpha=0.3)

# Denoised signal (remove highest frequency details)
denoised_coeffs = coeffs.copy()
denoised_coeffs[-1] = np.zeros_like(denoised_coeffs[-1])  # Remove finest detail
denoised_coeffs[-2] *= 0.5  # Attenuate next detail level
price_denoised = pywt.waverec(denoised_coeffs, wavelet)[:len(observed_price)]

axes[-1].plot(t, observed_price, 'gray', alpha=0.3, linewidth=0.5, label='Original')
axes[-1].plot(t, price_denoised, 'r-', linewidth=2, label='Denoised')
axes[-1].plot(t, true_price, 'g--', linewidth=1.5, alpha=0.7, label='True')
axes[-1].set_title('Wavelet Denoising')
axes[-1].set_xlabel('Time')
axes[-1].set_ylabel('Price')
axes[-1].legend()
axes[-1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Denoising performance
error_wavelet = np.mean((price_denoised - true_price)**2)
print(f"MSE Wavelet Denoising: {error_wavelet:.4f}")
print(f"Noise Reduction: {(1 - error_wavelet/np.var(noise))*100:.2f}%")

## 4. Kalman Filtering

In [ ]:
# Kalman filter for state-space modeling
# State: [position, velocity]
# Observation: position with noise

# Initialize Kalman filter
kf = KalmanFilter(
    transition_matrices=[[1, 1], [0, 1]],  # State transition
    observation_matrices=[[1, 0]],  # Observation model
    initial_state_mean=[observed_price[0], 0],
    initial_state_covariance=[[1, 0], [0, 1]],
    observation_covariance=[[0.5**2]],  # Measurement noise
    transition_covariance=[[0.01, 0], [0, 0.01]]  # Process noise
)

# Apply Kalman filter
state_means, state_covariances = kf.filter(observed_price)

# Extract filtered price and velocity
price_kalman = state_means[:, 0]
velocity_kalman = state_means[:, 1]

# Calculate confidence intervals
price_std = np.sqrt(state_covariances[:, 0, 0])

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Filtered price
axes[0].plot(t, observed_price, 'gray', alpha=0.3, linewidth=0.5, label='Observed')
axes[0].plot(t, price_kalman, 'b-', linewidth=2, label='Kalman Filtered')
axes[0].plot(t, true_price, 'r--', linewidth=1.5, alpha=0.7, label='True')
axes[0].fill_between(t, price_kalman - 2*price_std, price_kalman + 2*price_std,
                     alpha=0.2, color='blue', label='±2σ Confidence')
axes[0].set_ylabel('Price')
axes[0].set_title('Kalman Filter - Price Estimation')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Velocity (trend)
axes[1].plot(t, velocity_kalman, 'g-', linewidth=2)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Velocity')
axes[1].set_title('Estimated Price Velocity (Trend)')
axes[1].grid(True, alpha=0.3)

# Estimation error
error_kalman = price_kalman - true_price
axes[2].plot(t, error_kalman, 'r-', linewidth=1, alpha=0.7)
axes[2].axhline(y=0, color='black', linestyle='--')
axes[2].fill_between(t, -2*price_std, 2*price_std, alpha=0.2, color='gray')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Error')
axes[2].set_title('Estimation Error')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Performance metrics
mse_kalman = np.mean((price_kalman - true_price)**2)
print(f"MSE Kalman Filter: {mse_kalman:.4f}")
print(f"Mean Absolute Error: {np.mean(np.abs(error_kalman)):.4f}")
print(f"Average Velocity: {velocity_kalman.mean():.6f}")

## 5. Feature Extraction

In [ ]:
# Extract multiple features from price data
def extract_features(price, window=20):
    features = pd.DataFrame(index=range(len(price)))
    
    # Moving averages
    features['sma_fast'] = pd.Series(price).rolling(window=10).mean()
    features['sma_slow'] = pd.Series(price).rolling(window=50).mean()
    features['ema'] = pd.Series(price).ewm(span=window).mean()
    
    # Volatility
    returns = np.diff(price, prepend=price[0]) / price
    features['volatility'] = pd.Series(returns).rolling(window=window).std()
    
    # RSI (Relative Strength Index)
    delta = pd.Series(returns)
    gain = delta.where(delta > 0, 0).rolling(window=14).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=14).mean()
    rs = gain / loss
    features['rsi'] = 100 - (100 / (1 + rs))
    
    # MACD
    ema_fast = pd.Series(price).ewm(span=12).mean()
    ema_slow = pd.Series(price).ewm(span=26).mean()
    features['macd'] = ema_fast - ema_slow
    features['macd_signal'] = features['macd'].ewm(span=9).mean()
    
    # Bollinger Bands
    sma = pd.Series(price).rolling(window=window).mean()
    std = pd.Series(price).rolling(window=window).std()
    features['bb_upper'] = sma + 2*std
    features['bb_lower'] = sma - 2*std
    features['bb_width'] = (features['bb_upper'] - features['bb_lower']) / sma
    
    # Momentum
    features['momentum'] = pd.Series(price).pct_change(periods=10)
    
    # Statistical features
    features['skewness'] = pd.Series(returns).rolling(window=window).skew()
    features['kurtosis'] = pd.Series(returns).rolling(window=window).kurt()
    
    return features

# Extract features
features = extract_features(observed_price)

# Visualization
fig, axes = plt.subplots(4, 2, figsize=(16, 16))

# Price with moving averages
axes[0, 0].plot(t, observed_price, 'gray', alpha=0.3, linewidth=0.5, label='Price')
axes[0, 0].plot(t, features['sma_fast'], 'b-', linewidth=1.5, label='SMA Fast (10)')
axes[0, 0].plot(t, features['sma_slow'], 'r-', linewidth=1.5, label='SMA Slow (50)')
axes[0, 0].set_title('Moving Averages')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Bollinger Bands
axes[0, 1].plot(t, observed_price, 'b-', linewidth=1, label='Price')
axes[0, 1].plot(t, features['bb_upper'], 'r--', linewidth=1, label='Upper Band')
axes[0, 1].plot(t, features['bb_lower'], 'g--', linewidth=1, label='Lower Band')
axes[0, 1].fill_between(t, features['bb_lower'], features['bb_upper'], 
                        alpha=0.1, color='gray')
axes[0, 1].set_title('Bollinger Bands')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Volatility
axes[1, 0].plot(t, features['volatility'], 'orange', linewidth=1.5)
axes[1, 0].set_title('Rolling Volatility (20-period)')
axes[1, 0].set_ylabel('Volatility')
axes[1, 0].grid(True, alpha=0.3)

# RSI
axes[1, 1].plot(t, features['rsi'], 'purple', linewidth=1.5)
axes[1, 1].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought')
axes[1, 1].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold')
axes[1, 1].set_title('Relative Strength Index (RSI)')
axes[1, 1].set_ylabel('RSI')
axes[1, 1].set_ylim(0, 100)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# MACD
axes[2, 0].plot(t, features['macd'], 'b-', linewidth=1.5, label='MACD')
axes[2, 0].plot(t, features['macd_signal'], 'r-', linewidth=1.5, label='Signal')
axes[2, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[2, 0].set_title('MACD')
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)

# Momentum
axes[2, 1].plot(t, features['momentum'], 'green', linewidth=1)
axes[2, 1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[2, 1].set_title('Momentum (10-period)')
axes[2, 1].set_ylabel('Momentum')
axes[2, 1].grid(True, alpha=0.3)

# Skewness
axes[3, 0].plot(t, features['skewness'], 'brown', linewidth=1)
axes[3, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[3, 0].set_title('Rolling Skewness')
axes[3, 0].set_xlabel('Time')
axes[3, 0].set_ylabel('Skewness')
axes[3, 0].grid(True, alpha=0.3)

# Kurtosis
axes[3, 1].plot(t, features['kurtosis'], 'teal', linewidth=1)
axes[3, 1].axhline(y=0, color='black', linestyle='--', alpha=0.5)
axes[3, 1].set_title('Rolling Kurtosis')
axes[3, 1].set_xlabel('Time')
axes[3, 1].set_ylabel('Kurtosis')
axes[3, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFeature Statistics:")
print(features.describe().round(4))

## 6. Power Spectral Density

In [ ]:
# Compute power spectral density using Welch's method
frequencies, psd = welch(observed_price, fs=sampling_rate, nperseg=256)

# Find dominant frequencies
dominant_idx = np.argsort(psd)[-5:][::-1]  # Top 5
dominant_freqs = frequencies[dominant_idx]
dominant_power = psd[dominant_idx]

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Power spectral density
axes[0].semilogy(frequencies, psd, 'b-', linewidth=1.5)
for freq, power in zip(dominant_freqs, dominant_power):
    axes[0].plot(freq, power, 'ro', markersize=10)
    axes[0].text(freq, power, f'{freq:.2f} Hz', rotation=45, va='bottom')
axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Power Spectral Density')
axes[0].set_title('Power Spectral Density (Welch Method)')
axes[0].grid(True, alpha=0.3)

# Spectrogram
f, t_spec, Sxx = signal.spectrogram(observed_price, fs=sampling_rate, nperseg=128)
axes[1].pcolormesh(t_spec, f, 10 * np.log10(Sxx), shading='gouraud', cmap='viridis')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_xlabel('Time')
axes[1].set_title('Spectrogram')
axes[1].set_ylim(0, sampling_rate/4)  # Show up to Nyquist/2

plt.colorbar(axes[1].collections[0], ax=axes[1], label='Power (dB)')
plt.tight_layout()
plt.show()

print("\nDominant Frequencies:")
for i, (freq, power) in enumerate(zip(dominant_freqs, dominant_power), 1):
    print(f"  {i}. {freq:.4f} Hz (Power: {power:.2e})")

## 7. Summary

### Key Techniques:

1. **Digital Filtering**
   - Butterworth low-pass filter for noise reduction
   - Savitzky-Golay filter for peak preservation
   - Band-pass filtering for frequency isolation

2. **Wavelet Analysis**
   - Multi-resolution decomposition
   - Time-frequency localization
   - Adaptive denoising

3. **Kalman Filtering**
   - Optimal state estimation
   - Trend and velocity extraction
   - Real-time signal processing

4. **Feature Engineering**
   - Technical indicators (MA, RSI, MACD, Bollinger Bands)
   - Statistical features (volatility, skewness, kurtosis)
   - Momentum indicators

5. **Spectral Analysis**
   - Power spectral density estimation
   - Dominant frequency detection
   - Time-frequency analysis (spectrograms)

### Applications in Trading:
- Noise reduction for cleaner signals
- Trend identification and prediction
- Market cycle detection
- Feature generation for machine learning models
- Real-time signal quality monitoring